# Atelier Scikit-learn

Une entreprise possède plusieurs bâtiments équipés de capteurs IoT. Chaque capteur collecte régulièrement des informations sur la température, l'humidité, la pression, la consommation énergétique, le bâtiment, la date et l'heure de la mesure.  

Chaque mesure possède également un état (OK, ALERTE et ERREUR). L'objectif de l'atelier est de construire un modèle capable de prédire automatiquement l'état d'un capteur à partir de ses mesures. 

L'atelier suivra le workflow classique du Machine Learning : Dataset → Chargement → Exploration → Nettoyage → X / y → Train / Test → Prétraitement → Modèle → fit()→ predict()→ Évaluation → Sauvegarde → Chargement → Réutilisation 

## Partie 0 – mise en place de l’environnement 

**Objectif** : préparer l'environnement de travail en important les librairies nécessaires (manipulation de données et visualisation), avant de charger et explorer le dataset.

In [ ]:
#Installer et importer seaborn, matplotlib et pandas si pas encore
#%pip install pandas seaborn matplotlib scikit-learn

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Pour un affichage plus agréable des graphiques dans le notebook
%matplotlib inline

### Importer mesures_capteurs.csv dans le dataframe df et Explorer le dataframe df 

In [2]:
df = pd.read_csv("../data/mesures_capteurs.csv")

**Explication** : `pd.read_csv()` lit le fichier CSV situé dans `data/` et le charge dans un DataFrame Pandas nommé `df`. Le chemin `../data/mesures_capteurs.csv` remonte d'un niveau (depuis `notebooks/`) puis entre dans `data/`, ce qui suppose que le notebook est exécuté depuis le dossier `notebooks/`.

**Résultat** : le dataset est chargé en mémoire sous forme de tableau, prêt à être exploré.

In [3]:
df.head()

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


**Explication** : `df.head()` affiche par défaut les 5 premières lignes du DataFrame, pour avoir un premier aperçu visuel des données.

**Résultat** : on voit les colonnes `id_mesure, date_heure, id_capteur, batiment, temperature, humidite, pression, consommation, etat`, avec des valeurs cohérentes (ex. température autour de 20-30°C, état "OK").

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_mesure     605 non-null    str    
 1   date_heure    605 non-null    str    
 2   id_capteur    605 non-null    str    
 3   batiment      605 non-null    str    
 4   temperature   599 non-null    float64
 5   humidite      600 non-null    float64
 6   pression      600 non-null    float64
 7   consommation  600 non-null    float64
 8   etat          601 non-null    str    
dtypes: float64(4), str(5)
memory usage: 42.7 KB


**Explication** : `df.info()` affiche pour chaque colonne son type (`object` pour du texte, `float64` pour des nombres décimaux) ainsi que le nombre de valeurs non nulles. Cela permet de repérer immédiatement les valeurs manquantes par colonne.

**Résultat** : le dataset contient 605 lignes. Les colonnes `temperature`, `humidite`, `pression`, `consommation` et `etat` contiennent chacune quelques valeurs manquantes (moins de 600 valeurs non nulles sur 605).

In [5]:
df.describe()

,temperature,humidite,pression,consommation
count,599.000000,600.00000,600.000000,600.000000
mean,24.878314,64.92620,1012.221900,208.675417
std,4.059576,10.76905,10.599042,72.243567
min,-18.500000,28.52000,850.000000,18.120000
25%,22.570000,58.17250,1006.790000,160.177500
50%,24.860000,65.37500,1012.855000,206.150000
75%,27.275000,71.61500,1017.827500,254.127500
max,58.700000,145.00000,1038.430000,875.000000


In [6]:
df.shape

(605, 9)

**Explication** : `df.shape` retourne un tuple (nombre de lignes, nombre de colonnes).

**Résultat** : le dataset contient **605 lignes** et **9 colonnes**.

In [7]:
df.columns

Index(['id_mesure', 'date_heure', 'id_capteur', 'batiment', 'temperature',
       'humidite', 'pression', 'consommation', 'etat'],
      dtype='str')

**Explication** : `df.columns` liste les noms de toutes les colonnes du DataFrame.

**Résultat** : `id_mesure, date_heure, id_capteur, batiment, temperature, humidite, pression, consommation, etat`.

In [8]:
df["etat"].value_counts(dropna=False)

etat
OK        567
ALERTE     29
ERREUR      5
NaN         4
Name: count, dtype: int64

**Explication** : `value_counts()` compte le nombre d'occurrences de chaque valeur unique de la colonne `etat` (la cible qu'on cherche à prédire). Le paramètre `dropna=False` inclut aussi le comptage des valeurs manquantes (`NaN`).

**Résultat** : la colonne `etat` contient 567 mesures "OK", 29 "ALERTE", 5 "ERREUR", et 4 valeurs manquantes.

## Partie 1 – Gestion des doublons 

**Objectif** : vérifier si le dataset contient des lignes strictement identiques (doublons), qui pourraient fausser l'entraînement du modèle en sur-représentant artificiellement certaines observations, puis les supprimer le cas échéant.

### 1) vérifier l’existence de doublons dans df 

In [9]:
print("Nombre de doublons :", df.duplicated().sum())
df[df.duplicated()]

Nombre de doublons : 5


,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
183,M0599,2026-01-29 22:00:00,C011,B004,22.02,68.09,1005.90,227.81,OK
231,M0026,2026-01-06 01:00:00,C002,B001,22.87,77.99,1010.15,213.19,OK
355,M0147,2026-01-11 02:00:00,C003,B001,26.14,84.97,1003.59,142.31,OK
538,M0456,2026-01-23 23:00:00,C012,B004,20.68,72.69,1022.77,309.01,OK
539,M0302,2026-01-17 13:00:00,C002,B001,22.05,58.26,1007.30,140.42,OK


**Explication** : `df.duplicated()` retourne une série de booléens (`True` si la ligne est identique à une ligne déjà rencontrée précédemment, `False` sinon). `.sum()` additionne les `True` (comptés comme 1), donnant ainsi le nombre total de doublons. `df[df.duplicated()]` permet d'afficher ces lignes dupliquées.

**Résultat** : le dataset contient **5 doublons**.

### 2) le cas échéant, supprimer les doublons puis vérifier la suppression  

In [10]:
print("Shape avant suppression :", df.shape)

df = df.drop_duplicates()

print("Shape après suppression :", df.shape)
print("Doublons restants :", df.duplicated().sum())

Shape avant suppression : (605, 9)
Shape après suppression : (600, 9)
Doublons restants : 0


**Explication** : `df.drop_duplicates()` supprime les lignes dupliquées en ne conservant que la première occurrence de chaque ligne identique. On revérifie ensuite avec `duplicated().sum()` pour confirmer qu'il n'en reste plus.

**Résultat** : le dataset passe de **605 à 600 lignes** après suppression des 5 doublons. La vérification confirme **0 doublon restant**.

## Partie 2 – Sélection de y (cible) et X (caractéristiques)

**Objectif** : séparer le dataset en deux ensembles — `X`, les variables explicatives (caractéristiques) qui serviront à faire la prédiction, et `y`, la variable cible (ce qu'on cherche à prédire).

### 1) Définir "etat" comme la cible ou valeur à prédire et "temperature", "humidite", "pression" et "consommation" comme caractéristiques ou variables explicatives 

In [12]:
X = df[["temperature", "humidite", "pression", "consommation"]]
y = df["etat"]

### 2) Afficher les cinq premières lignes de X et de y

In [13]:
print(X.head())
print()
print(y.head())

   temperature  humidite  pression  consommation
0        25.46     58.06   1008.95        287.28
1        24.00     79.73    993.39        116.20
2        25.82     54.47   1010.32        288.50
3        28.23     69.39   1019.62        136.65
4        20.58     53.80   1016.58        182.62

0    OK
1    OK
2    OK
3    OK
4    OK
Name: etat, dtype: str


**Explication** : `df[[...]]` avec une liste de colonnes sélectionne plusieurs colonnes et retourne un DataFrame — ce sont les 4 caractéristiques (`temperature`, `humidite`, `pression`, `consommation`) qui vont permettre au modèle de faire ses prédictions. `df["etat"]` avec une seule colonne entre crochets simples retourne une Series, c'est la cible que le modèle doit apprendre à prédire.

**Résultat** : `X` contient 4 colonnes numériques (les mesures des capteurs), `y` contient la colonne `etat` avec des valeurs comme "OK".

### 3) Quel est le type du problème de machine learning ?  

Il s'agit d'un problème d'**apprentissage supervisé de classification**, et plus précisément une **classification multi-classes** (3 classes possibles : `OK`, `ALERTE`, `ERREUR`). On parle d'apprentissage supervisé car on dispose d'exemples déjà étiquetés (la colonne `etat` est connue dans les données d'entraînement), et de classification car la cible à prédire est une catégorie discrète (et non une valeur numérique continue, ce qui serait de la régression).

## Partie 3 – Découpage Train/Test

**Objectif** : diviser les données en un ensemble d'entraînement (train) et un ensemble de test (test), afin de pouvoir évaluer le modèle sur des données qu'il n'a jamais vues pendant l'entraînement.

**Prérequis** : `y` contient encore 4 valeurs manquantes (`etat` = NaN). Une ligne sans cible connue ne peut pas être imputée (contrairement à une variable explicative) ni utilisée pour l'entraînement ou le test, on la supprime donc avant le découpage.

In [16]:
print("Lignes avant suppression des etat manquants :", df.shape)

df = df.dropna(subset=["etat"])

print("Lignes après suppression des etat manquants :", df.shape)

# On redéfinit X et y sur le df nettoyé
X = df[["temperature", "humidite", "pression", "consommation"]]
y = df["etat"]

Lignes avant suppression des etat manquants : (596, 9)
Lignes après suppression des etat manquants : (596, 9)


**Explication** : `df.dropna(subset=["etat"])` supprime uniquement les lignes où la colonne `etat` est manquante (les autres valeurs manquantes, dans X, seront traitées séparément en Partie 4). On redéfinit ensuite X et y à partir de ce dataframe nettoyé.
**subset** : supprime une ligne seulement si la colonne etat est NaN, ignore les NaN dans les autres colonnes (temperature, humidite, etc.)". C'est important ici car on ne veut pas encore toucher aux valeurs manquantes de X, on veut uniquement se débarrasser des lignes où la cible y est inconnue.

**Résultat** : le dataset passe de 600 à **596 lignes** (4 lignes avec `etat` manquant supprimées).

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train :", X_train.shape)
print("X_test :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test :", y_test.shape)

X_train : (476, 4)
X_test : (120, 4)
y_train : (476,)
y_test : (120,)


In [20]:
print("Répartition dans y (données complètes) :")
print((y.value_counts(normalize=True) * 100).round(2))

print("\nRépartition dans y_train :")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nRépartition dans y_test :")
print((y_test.value_counts(normalize=True) * 100).round(2))

Répartition dans y (données complètes) :
etat
OK        94.30
ALERTE     4.87
ERREUR     0.84
Name: proportion, dtype: float64

Répartition dans y_train :
etat
OK        94.33
ALERTE     4.83
ERREUR     0.84
Name: proportion, dtype: float64

Répartition dans y_test :
etat
OK        94.17
ALERTE     5.00
ERREUR     0.83
Name: proportion, dtype: float64


**Explication** :

Le split est réalisé sur `X` et `y` (596 lignes restantes après suppression des `etat` manquants), avec :

- **`test_size=0.2`** : réserve 20 % des lignes pour l'ensemble de test, et donc 80 % pour l'ensemble d'entraînement.
- **`random_state=42`** : fixe la graine du générateur aléatoire utilisé pour choisir les lignes. Sans ce paramètre, le découpage serait différent à chaque exécution ; avec `random_state=42`, il est identique à chaque fois, ce qui rend le résultat reproductible.
- **`stratify=y`** : impose que la proportion de chaque classe (`OK`, `ALERTE`, `ERREUR`) soit respectée à l'identique dans `y_train` et dans `y_test`, plutôt que de tirer les lignes au hasard. C'est essentiel ici car les classes sont très déséquilibrées (94 % de `OK`) — sans stratification, l'ensemble de test pourrait par exemple ne contenir aucune ligne `ERREUR`.

La vérification des proportions est faite avec la fonction `value_counts(normalize=True)`, qui calcule la proportion de chaque classe (nombre d'occurrences ÷ total des lignes) au lieu d'un simple compte brut. On multiplie par 100 pour obtenir un pourcentage.

**Résultat** :

`X_train` contient 476 lignes, `X_test` 120 lignes (soit bien 80 % / 20 % de 596 lignes).

- Données complètes : OK 94.30 % / ALERTE 4.87 % / ERREUR 0.84 %
- Train : OK 94.33 % / ALERTE 4.83 % / ERREUR 0.84 %
- Test : OK 94.17 % / ALERTE 5.00 % / ERREUR 0.83 %

Les écarts entre les trois répartitions sont minimes (moins de 0.2 point de pourcentage), ce qui confirme que le train et le test sont bien représentatifs de la distribution originale des classes.

## Partie 4 – Gestion des valeurs manquantes 

**Objectif** : détecter les valeurs manquantes restantes dans les variables explicatives (X) et les remplacer par une valeur statistique cohérente, afin que le modèle puisse s'entraîner (la plupart des algorithmes de scikit-learn n'acceptent pas de NaN en entrée).

### 1) Vérifier l’existence de valeurs manquantes

In [21]:
print("Valeurs manquantes dans X_train :")
print(X_train.isna().sum())

print("\nValeurs manquantes dans X_test :")
print(X_test.isna().sum())

Valeurs manquantes dans X_train :
temperature     5
humidite        4
pression        5
consommation    3
dtype: int64

Valeurs manquantes dans X_test :
temperature     1
humidite        1
pression        0
consommation    2
dtype: int64


**Explication** : `isna()` retourne `True` pour chaque cellule manquante, `sum()` compte le nombre de `True` par colonne.

**Résultat** : X_train contient des valeurs manquantes dans `temperature` (5), `humidite` (4), `pression` (5) et `consommation` (3). X_test en contient aussi (`temperature` : 1, `humidite` : 1, `consommation` : 2, `pression` : 0).

###  2) Sélectionner SimpleImputer avec la médiane

In [22]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

**Explication** : `SimpleImputer` est un outil de scikit-learn qui remplace les valeurs manquantes par une statistique calculée sur les données. `strategy="median"` indique qu'on utilisera la **médiane** de chaque colonne pour combler ses valeurs manquantes.

###  3) Qu’est ce qui justifie le choix de la médiane ?

La médiane est plus **robuste aux valeurs extrêmes (outliers)** que la moyenne. Une valeur aberrante très élevée ou très basse peut fortement déformer la moyenne, alors que la médiane (la valeur du milieu une fois les données triées) n'est pas affectée par l'ampleur des valeurs extrêmes, seulement par leur position. Pour des mesures de capteurs IoT, où des erreurs de mesure ponctuelles (température ou humidité incohérentes) sont plausibles, la médiane donne une estimation plus fiable de la "valeur typique" que la moyenne.

###  4) Trouver les paramètres (médianes) de l’imputeur sur X_train

In [24]:
# Apprentissage des médianes sur X_train uniquement
imputer.fit(X_train)

print("Médianes apprises :")
print(dict(zip(X_train.columns, imputer.statistics_)))

Médianes apprises :
{'temperature': np.float64(24.9), 'humidite': np.float64(65.38), 'pression': np.float64(1012.3), 'consommation': np.float64(206.59)}


**Explication** : `imputer.fit(X_train)` calcule et mémorise la médiane de chaque colonne, **en se basant uniquement sur X_train**. C'est important : on n'utilise jamais X_test pour "apprendre" quoi que ce soit, afin d'éviter une fuite de données (data leakage) qui fausserait l'évaluation du modèle.

**Résultat** : les médianes apprises sont : température ≈ 24.9, humidité ≈ 65.38, pression ≈ 1012.3, consommation ≈ 206.59.

###  5) Déterminer X_train_imputed et X_test_imputed, les transformés de X_train et X_test 

In [25]:
X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

# transform() retourne un tableau numpy, on le reconvertit en DataFrame
X_train_imputed = pd.DataFrame(X_train_imputed, columns=X_train.columns, index=X_train.index)
X_test_imputed = pd.DataFrame(X_test_imputed, columns=X_test.columns, index=X_test.index)

print("NaN restants dans X_train_imputed :", X_train_imputed.isna().sum().sum())
print("NaN restants dans X_test_imputed :", X_test_imputed.isna().sum().sum())

NaN restants dans X_train_imputed : 0
NaN restants dans X_test_imputed : 0


**Explication** : `imputer.transform(...)` applique les médianes apprises à l'étape précédente (sur X_train) pour remplacer les valeurs manquantes — **dans X_train comme dans X_test**, avec les mêmes médianes (celles de X_train). On reconvertit le résultat (un tableau numpy) en DataFrame pour conserver les noms de colonnes et les index d'origine.

**Résultat** : `X_train_imputed` et `X_test_imputed` ne contiennent plus aucune valeur manquante (0 NaN dans les deux).